# Visualize Input Data Layers

Run cells **from top to bottom**. The raster cell both lists files **and draws a map**.

To see a different layer, edit `category` / `layer` in that cell and run it again.

In [ ]:
import importlib.util
import subprocess
import sys

needed = [
    "geopandas",
    "matplotlib",
    "rioxarray",
    "rasterio",
    "contextily",
    "pandas",
    "numpy",
]
missing = [pkg for pkg in needed if importlib.util.find_spec(pkg) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All visualization packages are already installed.")

In [ ]:
%matplotlib inline

from pathlib import Path
import warnings

import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray as rxr
from IPython.display import display
from mpl_toolkits.axes_grid1 import make_axes_locatable

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titlesize"] = 12


def show_figure(fig):
    display(fig)
    plt.show()

## Find input layers

In [ ]:
def find_input_dir() -> Path:
    here = Path.cwd().resolve()
    candidates = [
        here / "Files needed for runing" / "Input_Data_Layers",
        here / "MPM_Curnamona_REE" / "Files needed for runing" / "Input_Data_Layers",
        here.parent / "MPM_Curnamona_REE" / "Files needed for runing" / "Input_Data_Layers",
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        "Could not find Input_Data_Layers. Open the notebook from the Geology folder "
        "or MPM_Curnamona_REE, then re-run this cell."
    )

INPUT_DIR = find_input_dir()
print("Input directory:", INPUT_DIR)

## Study area and REE occurrences

In [ ]:
frame = gpd.read_file(INPUT_DIR / "GIS" / "Clip_frame.shp")
min_occ = gpd.read_file(INPUT_DIR / "GIS" / "ree.shp")

if frame.crs is None:
    frame = frame.set_crs("EPSG:4283")
if min_occ.crs is None:
    min_occ = min_occ.set_crs(frame.crs)
elif min_occ.crs != frame.crs:
    min_occ = min_occ.to_crs(frame.crs)

bounds = frame.total_bounds
extent = [bounds[0], bounds[2], bounds[1], bounds[3]]
CRS_EPSG = frame.crs.to_epsg() or 4283


def add_basemap(ax):
    try:
        cx.add_basemap(ax, crs=frame.crs, source=cx.providers.Esri.WorldGrayCanvas)
    except Exception as exc:
        print(f"Basemap skipped: {exc}")


print("CRS:", frame.crs)
print("REE occurrences:", len(min_occ))

fig, ax = plt.subplots(figsize=(8, 8))
min_occ.plot(ax=ax, color="gold", edgecolor="black", markersize=40, label="REE occurrences")
frame.plot(ax=ax, edgecolor="red", facecolor="none", linewidth=2, label="Study area")
add_basemap(ax)
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3])
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Study area and REE mineral occurrences")
ax.legend(loc="lower right")
ax.set_aspect("equal")
show_figure(fig)

## Rasters (plots every GeoTIFF)

This cell lists the files, then draws **all** rasters in one gallery per category (Magnetics, Gravity, Radiometrics, Remote Sensing, DEM).

In [ ]:
RASTER_CATEGORIES = {
    "Magnetics": INPUT_DIR / "Magnetics",
    "Gravity": INPUT_DIR / "Gravity",
    "Radiometrics": INPUT_DIR / "Radiometrics",
    "Remote Sensing": INPUT_DIR / "Remote Sensing",
    "DEM": INPUT_DIR / "DEM",
}

raster_catalog = {
    name: sorted(folder.glob("*.tif"))
    for name, folder in RASTER_CATEGORIES.items()
    if folder.exists()
}

print("Available rasters:")
for name, files in raster_catalog.items():
    print(f"  {name}: {[p.stem for p in files]}")


def load_raster(path: Path):
    raster = rxr.open_rasterio(path, masked=True).squeeze()
    if raster.rio.crs is None:
        raster = raster.rio.write_crs(frame.crs)
    elif raster.rio.crs.to_epsg() != CRS_EPSG:
        raster = raster.rio.reproject(frame.crs)
    return raster


def draw_raster_on_ax(ax, path: Path, show_occurrences: bool = False, colorbar: bool = True):
    raster = load_raster(path)
    values = np.asarray(raster.values, dtype=float)
    finite = np.isfinite(values)
    if not finite.any():
        ax.set_title(f"{path.stem} (no data)")
        ax.axis("off")
        return

    v_mean = np.nanmean(values)
    v_std = np.nanstd(values)
    vmin, vmax = v_mean - 2 * v_std, v_mean + 2 * v_std
    if vmin == vmax:
        vmin, vmax = np.nanmin(values), np.nanmax(values)

    left, bottom, right, top = raster.rio.bounds()
    image = ax.imshow(
        values,
        cmap="Spectral_r",
        extent=(left, right, bottom, top),
        vmin=vmin,
        vmax=vmax,
        origin="upper",
        interpolation="nearest",
    )
    frame.plot(ax=ax, edgecolor="red", facecolor="none", linewidth=0.8)
    if show_occurrences:
        min_occ.plot(ax=ax, color="gold", edgecolor="black", markersize=8, linewidth=0.3)
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_title(path.stem.replace("_", " "), fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    if colorbar:
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="4%", pad=0.04)
        ax.figure.colorbar(image, cax=cax)


def plot_category_gallery(category: str, show_occurrences: bool = False, ncols: int = 3):
    files = raster_catalog[category]
    n = len(files)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.4 * ncols, 4.0 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, path in zip(axes, files):
        draw_raster_on_ax(ax, path, show_occurrences=show_occurrences, colorbar=True)
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle(f"{category} ({n} layers)", fontsize=14, y=1.01)
    fig.tight_layout()
    show_figure(fig)


print("\nPlotting all rasters...")
for category in raster_catalog:
    plot_category_gallery(category, show_occurrences=False)
print("Done.")


## Geology

In [ ]:
geology_dir = INPUT_DIR / "Geology factors" / "original"
geology_layers = {}
for path in sorted(geology_dir.glob("*.shp")):
    gdf = gpd.read_file(path)
    if gdf.crs is None:
        gdf = gdf.set_crs(frame.crs)
    elif gdf.crs != frame.crs:
        gdf = gdf.to_crs(frame.crs)
    geology_layers[path.stem] = gdf

GEOLOGY_STYLE = {
    "Archean_Early_Mesoproterozoic_Faults": dict(color="black", linewidth=0.8, facecolor="none"),
    "Felsic_granite": dict(color="#d95f02", linewidth=0.4, facecolor="#d95f02", alpha=0.45),
    "Mesoproterozoic": dict(color="#7570b3", linewidth=0.4, facecolor="#7570b3", alpha=0.35),
}

print("Geology layers:", list(geology_layers))
layer = list(geology_layers)[0]

gdf = geology_layers[layer]
style = GEOLOGY_STYLE.get(layer, dict(color="navy", linewidth=0.6, facecolor="none"))
fig, ax = plt.subplots(figsize=(8, 8))
add_basemap(ax)
gdf.plot(
    ax=ax,
    edgecolor=style["color"],
    facecolor=style.get("facecolor", "none"),
    linewidth=style.get("linewidth", 0.6),
    alpha=style.get("alpha", 1),
)
frame.plot(ax=ax, edgecolor="red", facecolor="none", linewidth=2)
min_occ.plot(ax=ax, color="gold", edgecolor="black", markersize=28)
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3])
ax.set_title(layer.replace("_", " "))
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
show_figure(fig)

fig, ax = plt.subplots(figsize=(9, 9))
add_basemap(ax)
for name, gdf in geology_layers.items():
    style = GEOLOGY_STYLE.get(name, dict(color="navy", linewidth=0.6, facecolor="none"))
    gdf.plot(
        ax=ax,
        edgecolor=style["color"],
        facecolor=style.get("facecolor", "none"),
        linewidth=style.get("linewidth", 0.6),
        alpha=style.get("alpha", 1),
        label=name.replace("_", " "),
    )
frame.plot(ax=ax, edgecolor="red", facecolor="none", linewidth=2, label="Study area")
min_occ.plot(ax=ax, color="gold", edgecolor="black", markersize=32, label="REE occurrences")
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3])
ax.set_title("Geology overlay with REE occurrences")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(loc="lower right", fontsize=8)
ax.set_aspect("equal")
show_figure(fig)

## Geochemistry

Edit `element` (for example `La`, `Y`, `Th`, `U3O8`) and re-run this cell.

In [ ]:
geochem_shp = gpd.read_file(INPUT_DIR / "Geochemical" / "geochemical_unlabel_all_869_XY.shp")
if geochem_shp.crs is None:
    geochem_shp = geochem_shp.set_crs(frame.crs)
else:
    geochem_shp = geochem_shp.to_crs(frame.crs)

geochem_csv = pd.read_csv(INPUT_DIR / "Geochemical" / "ILRRPCA.csv")
lon_col, lat_col = "lon_gda94", "lat_gda94"
element_cols = [c for c in geochem_csv.columns if c not in {lon_col, lat_col}]
preferred = [c for c in ["La", "Y", "Th", "U3O8", "Ce", "Nd"] if c in element_cols]
element_options = preferred + [c for c in element_cols if c not in preferred]

print("CSV samples:", len(geochem_csv))
print("Elements:", element_options)

fig, ax = plt.subplots(figsize=(8, 8))
add_basemap(ax)
geochem_shp.plot(ax=ax, color="steelblue", markersize=12, alpha=0.8, label="Geochemical samples")
min_occ.plot(ax=ax, color="gold", edgecolor="black", markersize=32, label="REE occurrences")
frame.plot(ax=ax, edgecolor="red", facecolor="none", linewidth=2, label="Study area")
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3])
ax.set_title("Geochemical sample locations")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(loc="lower right")
ax.set_aspect("equal")
show_figure(fig)

element = element_options[0]
print("Plotting element:", element)

fig, ax = plt.subplots(figsize=(8, 8))
add_basemap(ax)
sc = ax.scatter(
    geochem_csv[lon_col],
    geochem_csv[lat_col],
    c=geochem_csv[element],
    cmap="plasma",
    s=18,
    edgecolors="none",
    zorder=3,
)
frame.plot(ax=ax, edgecolor="red", facecolor="none", linewidth=2, zorder=4)
min_occ.plot(ax=ax, color="none", edgecolor="lime", markersize=40, linewidth=1.2, zorder=5)
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3])
ax.set_title(f"Geochemistry: {element}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="4%", pad=0.08)
fig.colorbar(sc, cax=cax, label=element)
show_figure(fig)